# Assemble the global ISMN + ERA5 + SMAP + static + spatial-index dataset

This is a **low-memory, disk-backed merge notebook** based on your uploaded schema report. Albedo is skipped.

## Outputs

- `merged_sample_features.parquet`: one row per `sample_id`; candidate metadata, ERA5 scalars, SMAP scalars/quality flag, static soil and terrain, Sentinel-1 cluster metadata, and row indices only where a verified index exists.
- `global_target_observations.parquet`: original daily ISMN measurement records with `sample_id` attached where station/network/date matches. It preserves sensor/depth observations instead of silently averaging across depths.
- `spatial_alignment_manifest.parquet`: one row per sample containing yearly MODIS/Sentinel-1 Zarr paths and row indices. Sentinel-1 row indices are validated against the yearly index; MODIS row indices are marked as based on a sidecar row-order assumption.
- `global_merge_validation.json`: join QA, row counts, coverage, and Zarr array inventory.

## Safety decisions

- `station_date_candidates.parquet` is the canonical index (3,514,643 rows per the report).
- ERA5 and SMAP are joined by `sample_id`, after uniqueness and station/date checks.
- Static CSVs are joined by `station_key`.
- Daily ISMN measurements can contain multiple sensor/depth rows for the same station/date. They are preserved in a separate long-format target table; no target averaging rule is assumed.
- Zarr patches remain in Zarr. Sentinel-1 uses its year-specific `samples_nearest_valid_YEAR.parquet` row index. The MODIS sidecar does not expose a row index, so its mapping is constructed from sidecar row order only as an explicit assumption; matching row counts and sample identities do **not** independently prove that the Zarr patch order is identical. Confirm the MODIS writer order before treating those patch indices as verified.
- DuckDB has a memory cap and can spill to disk. It does not load full Parquets or patches into pandas.

Run cells from top to bottom. Existing outputs are backed up before replacement.

In [1]:
from pathlib import Path
from datetime import datetime
import csv
import json
import shutil
import tempfile
import subprocess
import sys
import importlib.util

for package, module in [("duckdb", "duckdb"), ("pyarrow", "pyarrow"), ("zarr", "zarr"), ("numpy", "numpy")]:
    if importlib.util.find_spec(module) is None:
        print(f"Installing missing dependency: {package}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

import duckdb
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import zarr

ROOT = Path.cwd()
if ROOT.name != "Global" or not (ROOT / "station_date_candidates.parquet").exists():
    candidates = [Path.home() / "Code/UG_SM/Data/Global", Path.cwd() / "Data/Global"]
    ROOT = next((p for p in candidates if (p / "station_date_candidates.parquet").exists()), ROOT)

CANDIDATES = ROOT / "station_date_candidates.parquet"
DAILY = ROOT / "global_ismn_daily.parquet"
ERA5 = ROOT / "era5_backend/era5_by_sample.parquet"
SMAP = ROOT / "smap/smap_by_sample.parquet"
SOIL_CSV = ROOT / "Statics/global_static_soil.csv"
TERRAIN_CSV = ROOT / "Statics/global_static_terrain.csv"
S1_STATION_CLUSTERS = ROOT / "sentinel1/stations_with_clusters.parquet"
S1_ROOT = ROOT / "sentinel1/nearest_valid_overview_v1"
S1_INDEX_FILES = [S1_ROOT / f"samples_nearest_valid_{year}.parquet" for year in range(2015, 2027)]
MODIS_ROOTS = [
    ROOT / "modis/evi_lst_patch_dataset",
    ROOT / "modis/modis_patch_dataset",
    ROOT / "modis/ndvi_lst_patch_dataset",
]

FEATURES_OUT = ROOT / "merged_sample_features.parquet"
TARGETS_OUT = ROOT / "global_target_observations.parquet"
SPATIAL_OUT = ROOT / "spatial_alignment_manifest.parquet"
VALIDATION_OUT = ROOT / "global_merge_validation.json"

MEMORY_LIMIT = "4GB"   # lower to 512MB if needed
THREADS = 2              # lower to 1 if RAM is very tight
INDEX_BATCH_ROWS = 50_000

print("Root:", ROOT.resolve())
print("DuckDB:", duckdb.__version__, "| Zarr:", zarr.__version__, "| PyArrow:", pa.__version__)


Root: /home/mayur/Code/UG_SM/Data/Global
DuckDB: 1.5.6 | Zarr: 3.4.0 | PyArrow: 23.0.1


In [2]:
# Confirm required inputs before making outputs.
required = [CANDIDATES, DAILY, ERA5, SMAP, SOIL_CSV, TERRAIN_CSV, S1_STATION_CLUSTERS]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Required inputs missing:\n" + "\n".join(missing))

print("Required scalar/label inputs:")
for p in required:
    print(f"  {p.relative_to(ROOT)} ({p.stat().st_size:,} bytes)")
print("Optional Sentinel sample index files:")
for p in S1_INDEX_FILES:
    if p.exists():
        print(f"  {p.relative_to(ROOT)}")


Required scalar/label inputs:
  station_date_candidates.parquet (5,883,140 bytes)
  global_ismn_daily.parquet (60,840,090 bytes)
  era5_backend/era5_by_sample.parquet (95,951,021 bytes)
  smap/smap_by_sample.parquet (29,441,475 bytes)
  Statics/global_static_soil.csv (147,419 bytes)
  Statics/global_static_terrain.csv (183,905 bytes)
  sentinel1/stations_with_clusters.parquet (129,255 bytes)
Optional Sentinel sample index files:
  sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2015.parquet
  sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2016.parquet
  sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2017.parquet
  sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2018.parquet
  sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2019.parquet
  sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2020.parquet
  sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2021.parquet
  sentinel1/nearest_valid_overview_v1/samples_nearest_va

## 1. Inventory Zarr stores and discover explicit sample-ID arrays

The previous report recorded Zarr attributes but did not list nested arrays. This cell recursively records array names, shapes, dtypes, chunks, and attributes without reading patch values. If a store has a one-dimensional `sample_id`/`sample_ids` array, it streams that vector to a small Parquet index so its rows can be aligned by ID.

In [3]:
WORK_DIR = Path(tempfile.mkdtemp(prefix=".global_merge_work_", dir=ROOT))
INDEX_DIR = WORK_DIR / "spatial_id_indexes"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
DUCKDB_TEMP = WORK_DIR / "duckdb_spill"
DUCKDB_TEMP.mkdir(parents=True, exist_ok=True)


def attrs_dict(obj):
    try:
        attrs = obj.attrs
        if hasattr(attrs, "asdict"):
            return dict(attrs.asdict())
        return dict(attrs)
    except Exception:
        return {}


def group_members(group):
    """Return immediate (name, object) pairs compatible with common Zarr APIs."""
    try:
        members = list(group.members())
        if members:
            return members
    except Exception:
        pass
    items = []
    try:
        items.extend(list(group.groups()))
    except Exception:
        pass
    try:
        items.extend(list(group.arrays()))
    except Exception:
        pass
    return items


def walk_zarr(group, prefix=""):
    arrays = []
    for name, obj in group_members(group):
        full = f"{prefix}/{name}".strip("/")
        if hasattr(obj, "shape") and hasattr(obj, "dtype"):
            arrays.append({
                "name": full,
                "shape": list(obj.shape),
                "dtype": str(obj.dtype),
                "chunks": list(obj.chunks) if getattr(obj, "chunks", None) is not None else None,
                "attrs": attrs_dict(obj),
            })
        else:
            arrays.extend(walk_zarr(obj, full))
    return arrays


def zarr_stores_for_root(path):
    markers = ("zarr.json", ".zgroup", ".zarray")
    if not path.exists():
        return []
    if path.is_dir() and any((path / marker).exists() for marker in markers):
        return [path]
    stores = []
    try:
        children = sorted(path.iterdir())
    except OSError:
        return []
    for child in children:
        if not child.is_dir():
            continue
        if any((child / marker).exists() for marker in markers):
            stores.append(child)
        else:
            try:
                grandchildren = sorted(child.iterdir())
            except OSError:
                continue
            stores.extend(p for p in grandchildren if p.is_dir() and any((p / marker).exists() for marker in markers))
    return stores

zarr_inventory = []
for zroot in MODIS_ROOTS + [S1_ROOT]:
    stores = zarr_stores_for_root(zroot)
    if not stores:
        zarr_inventory.append({"root": str(zroot.relative_to(ROOT)), "error": "no Zarr stores found at root/child levels"})
        print("No Zarr stores found:", zroot.relative_to(ROOT))
        continue
    for store in stores:
        entry = {"root": str(zroot.relative_to(ROOT)), "path": str(store.relative_to(ROOT)), "attrs": {}, "arrays": [], "id_index_path": None}
        try:
            zg = zarr.open_group(str(store), mode="r")
            entry["attrs"] = attrs_dict(zg)
            entry["arrays"] = walk_zarr(zg)
            print(f"\n{store.relative_to(ROOT)} | attrs={entry['attrs']}")
            for a in entry["arrays"]:
                print(f"  {a['name']} shape={a['shape']} dtype={a['dtype']} chunks={a['chunks']}")
        except Exception as exc:
            entry["error"] = repr(exc)
            print("Could not open:", store, repr(exc))
            zarr_inventory.append(entry)
            continue

        id_arrays = [a for a in entry["arrays"] if a["name"].split("/")[-1].lower() in {"sample_id", "sample_ids", "sampleid", "sampleids"} and len(a["shape"]) == 1]
        if len(id_arrays) == 1:
            relname = id_arrays[0]["name"]
            arr = zg[relname]
            row_arrays = [a for a in entry["arrays"] if a["name"] != relname and a["shape"] and int(a["shape"][0]) == int(arr.shape[0])]
            if not row_arrays:
                entry["sample_id_index_note"] = "Found 1-D sample_id array, but no other array has a matching first dimension; row alignment not assumed."
                print("  Found sample IDs, but no patch/data array with matching row count; not auto-indexed.")
                zarr_inventory.append(entry)
                continue
            entry["sample_id_index_matches_arrays"] = [a["name"] for a in row_arrays]
            import re
            year_match = next((int(part) for part in store.parts if part.isdigit() and len(part) == 4), None)
            if year_match is None:
                match = re.search(r"(20\d{2})", store.name)
                year_match = int(match.group(1)) if match else None
            if year_match is None:
                entry["sample_id_index_note"] = "Found sample_id array but could not infer year; no mapping index written."
                zarr_inventory.append(entry)
                continue
            safe_name = "".join(ch if ch.isalnum() else "_" for ch in f"{zroot.name}_{year_match}")
            id_path = INDEX_DIR / f"{safe_name}_sample_ids.parquet"
            writer = None
            try:
                for start in range(0, arr.shape[0], INDEX_BATCH_ROWS):
                    stop = min(start + INDEX_BATCH_ROWS, arr.shape[0])
                    values = np.asarray(arr[start:stop])
                    table = pa.table({
                        "sample_id": pa.array(values),
                        "row_index": pa.array(np.arange(start, stop, dtype=np.int64)),
                        "year": pa.array([year_match] * (stop - start), type=pa.int32()),
                        "store_path": pa.array([str(store.relative_to(ROOT))] * (stop - start)),
                    })
                    if writer is None:
                        writer = pq.ParquetWriter(id_path, table.schema, compression="zstd")
                    writer.write_table(table)
            finally:
                if writer is not None:
                    writer.close()
            entry["id_index_path"] = str(id_path)
            print(f"  Streamed sample-ID index from {relname}: {arr.shape[0]:,} rows -> {id_path.name}")
        elif len(id_arrays) > 1:
            entry["sample_id_array_candidates"] = [a["name"] for a in id_arrays]
            print("  Multiple candidate sample_id arrays; not auto-indexed.")
        else:
            print("  No explicit 1-D sample_id array; row order will not be assumed.")
        zarr_inventory.append(entry)

# Persist inventory separately, even if a later merge check stops.
inventory_for_report = []
for entry in zarr_inventory:
    clean = dict(entry)
    temporary_index = clean.pop("id_index_path", None)
    if temporary_index:
        clean["sample_id_index_extracted"] = True
        clean["sample_id_index_storage"] = "temporary; removed after merge"
    inventory_for_report.append(clean)
(ROOT / "global_zarr_inventory.json").write_text(json.dumps(inventory_for_report, indent=2, default=str), encoding="utf-8")
print("\nZarr stores inventoried:", sum(1 for x in zarr_inventory if x.get("arrays") is not None and "error" not in x))



modis/evi_lst_patch_dataset/2015/patches.zarr | attrs={}
  evi shape=[329903, 27, 27] dtype=int16 chunks=[128, 27, 27]
  done shape=[329903] dtype=bool chunks=[4096]
  lst shape=[329903, 13, 13] dtype=uint16 chunks=[256, 13, 13]
  No explicit 1-D sample_id array; row order will not be assumed.

modis/evi_lst_patch_dataset/2016/patches.zarr | attrs={}
  done shape=[394790] dtype=bool chunks=[4096]
  evi shape=[394790, 27, 27] dtype=int16 chunks=[128, 27, 27]
  lst shape=[394790, 13, 13] dtype=uint16 chunks=[256, 13, 13]
  No explicit 1-D sample_id array; row order will not be assumed.

modis/evi_lst_patch_dataset/2017/patches.zarr | attrs={}
  evi shape=[369953, 27, 27] dtype=int16 chunks=[128, 27, 27]
  lst shape=[369953, 13, 13] dtype=uint16 chunks=[256, 13, 13]
  done shape=[369953] dtype=bool chunks=[4096]
  No explicit 1-D sample_id array; row order will not be assumed.

modis/evi_lst_patch_dataset/2018/patches.zarr | attrs={}
  lst shape=[344806, 13, 13] dtype=uint16 chunks=[256,

/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [4]:
# Open DuckDB with memory limit and disk spilling. Full source tables are not loaded into pandas.
con = duckdb.connect(str(WORK_DIR / "merge.duckdb"))
con.execute(f"SET memory_limit={repr(MEMORY_LIMIT)}")
con.execute(f"SET temp_directory={repr(str(DUCKDB_TEMP.resolve()))}")
con.execute(f"SET threads={int(THREADS)}")
con.execute("SET TimeZone='UTC'")


def sqlq(value):
    return "'" + str(value).replace("'", "''") + "'"


def parquet_rows(path):
    return con.execute(f"SELECT COUNT(*) FROM read_parquet({sqlq(path.resolve())})").fetchone()[0]

base_rows = parquet_rows(CANDIDATES)
base_ids = con.execute(f"SELECT COUNT(DISTINCT sample_id) FROM read_parquet({sqlq(CANDIDATES.resolve())})").fetchone()[0]
base_duplicate_sample_day_keys = con.execute(f"""
SELECT COUNT(*) FROM (
 SELECT network, station_id, CAST(sample_date AS DATE) AS sample_date
 FROM read_parquet({sqlq(CANDIDATES.resolve())})
 GROUP BY network, station_id, CAST(sample_date AS DATE)
 HAVING COUNT(*) > 1
)
""").fetchone()[0]

print(f"Canonical candidates: {base_rows:,} rows; {base_ids:,} distinct sample_id values")
print(f"Duplicate network/station/date groups in canonical index: {base_duplicate_sample_day_keys:,}")
if base_rows != base_ids:
    raise ValueError("sample_id is not unique in the canonical candidate table.")
if base_duplicate_sample_day_keys:
    raise ValueError("Canonical table has duplicate network/station/date groups; daily label matching would be ambiguous.")


Canonical candidates: 3,514,643 rows; 3,514,643 distinct sample_id values
Duplicate network/station/date groups in canonical index: 0


## 2. Validate scalar joins before writing

These checks stop if scalar joins can multiply rows. ERA5/SMAP use `sample_id`; static features use `station_key`; daily ISMN measurements join by network, station, and date.

In [5]:
sources = {"ERA5": ERA5, "SMAP": SMAP}
source_qc = {}
for label, path in sources.items():
    rows = parquet_rows(path)
    distinct_ids = con.execute(f"SELECT COUNT(DISTINCT sample_id) FROM read_parquet({sqlq(path.resolve())})").fetchone()[0]
    duplicate_groups = con.execute(f"""
        SELECT COUNT(*) FROM (
          SELECT sample_id FROM read_parquet({sqlq(path.resolve())})
          GROUP BY sample_id HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    missing_from_source = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet({sqlq(CANDIDATES.resolve())}) b
        LEFT JOIN read_parquet({sqlq(path.resolve())}) x USING (sample_id)
        WHERE x.sample_id IS NULL
    """).fetchone()[0]
    extra_in_source = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet({sqlq(path.resolve())}) x
        LEFT JOIN read_parquet({sqlq(CANDIDATES.resolve())}) b USING (sample_id)
        WHERE b.sample_id IS NULL
    """).fetchone()[0]
    source_qc[label] = {
        "rows": rows, "distinct_sample_ids": distinct_ids,
        "duplicate_sample_id_groups": duplicate_groups,
        "candidate_ids_missing_in_source": missing_from_source,
        "source_ids_not_in_candidates": extra_in_source,
    }
    print(label, source_qc[label])
    if duplicate_groups or rows != distinct_ids:
        raise ValueError(f"{label} sample_id is not unique; merge stopped before writing outputs.")

for label, path in sources.items():
    mismatch = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet({sqlq(CANDIDATES.resolve())}) b
        JOIN read_parquet({sqlq(path.resolve())}) x USING(sample_id)
        WHERE TRIM(CAST(b.station_id AS VARCHAR)) <> TRIM(CAST(x.station_id AS VARCHAR))
           OR CAST(b.sample_date AS DATE) <> CAST(x.sample_date AS DATE)
    """).fetchone()[0]
    source_qc[label]["sample_id_station_date_mismatches"] = mismatch
    print(f"{label} sample_id station/date mismatches: {mismatch:,}")
    if mismatch:
        raise ValueError(f"{label} has sample_id rows whose station/date mismatches the canonical index.")

# Static table views. Use the provided station_key, then verify it agrees with network::station_id.
con.execute(f"""
CREATE OR REPLACE VIEW static_soil AS
SELECT TRIM(CAST(station_key AS VARCHAR)) AS station_key,
       TRIM(CAST(network AS VARCHAR)) || '::' || TRIM(CAST(station_id AS VARCHAR)) AS derived_key,
       TRY_CAST(clay_pct AS DOUBLE) AS clay_pct,
       TRY_CAST(sand_pct AS DOUBLE) AS sand_pct,
       TRY_CAST(silt_pct AS DOUBLE) AS silt_pct
FROM read_csv_auto({sqlq(SOIL_CSV.resolve())}, header=true, all_varchar=true)
""")
con.execute(f"""
CREATE OR REPLACE VIEW static_terrain AS
SELECT TRIM(CAST(station_key AS VARCHAR)) AS station_key,
       TRIM(CAST(network AS VARCHAR)) || '::' || TRIM(CAST(station_id AS VARCHAR)) AS derived_key,
       TRY_CAST(elevation_m AS DOUBLE) AS elevation_m,
       TRY_CAST(slope_deg AS DOUBLE) AS slope_deg,
       TRY_CAST(aspect_deg AS DOUBLE) AS aspect_deg
FROM read_csv_auto({sqlq(TERRAIN_CSV.resolve())}, header=true, all_varchar=true)
""")
for label in ("soil", "terrain"):
    dup = con.execute(f"SELECT COUNT(*) FROM (SELECT station_key FROM static_{label} GROUP BY station_key HAVING COUNT(*) > 1)").fetchone()[0]
    mismatch = con.execute(f"SELECT COUNT(*) FROM static_{label} WHERE station_key <> derived_key OR station_key IS NULL OR station_key = ''").fetchone()[0]
    print(f"Static {label}: duplicate station keys={dup:,}; key inconsistencies={mismatch:,}")
    if dup or mismatch:
        raise ValueError(f"Static {label} keys are duplicated or disagree with network::station_id.")
    coverage = con.execute(f"""
      SELECT COUNT(DISTINCT TRIM(CAST(b.network AS VARCHAR)) || '::' || TRIM(CAST(b.station_id AS VARCHAR)))
      FROM read_parquet({sqlq(CANDIDATES.resolve())}) b
      JOIN static_{label} s
        ON s.station_key = TRIM(CAST(b.network AS VARCHAR)) || '::' || TRIM(CAST(b.station_id AS VARCHAR))
    """).fetchone()[0]
    print(f"Unique stations covered by static {label}: {coverage:,} / 2,210 in the schema report")


ERA5 {'rows': 3514643, 'distinct_sample_ids': 3514643, 'duplicate_sample_id_groups': 0, 'candidate_ids_missing_in_source': 0, 'source_ids_not_in_candidates': 0}
SMAP {'rows': 3514643, 'distinct_sample_ids': 3514643, 'duplicate_sample_id_groups': 0, 'candidate_ids_missing_in_source': 0, 'source_ids_not_in_candidates': 0}
ERA5 sample_id station/date mismatches: 0
SMAP sample_id station/date mismatches: 0
Static soil: duplicate station keys=0; key inconsistencies=0
Unique stations covered by static soil: 2,138 / 2,210 in the schema report
Static terrain: duplicate station keys=0; key inconsistencies=0
Unique stations covered by static terrain: 2,210 / 2,210 in the schema report


## 3. Create explicit sample-to-patch indexes and merge sample-level features

In [6]:
# ============================================================
# CELL 9: VALIDATE SPATIAL INDEXES + BUILD FEATURE TABLE
# ============================================================

import re

YEARS = list(range(2015, 2027))


def zarr_year_from_entry(entry):
    path = Path(entry["path"])

    for part in path.parts:
        if part.isdigit() and len(part) == 4:
            return int(part)

    match = re.search(r"(20\d{2})", path.name)
    return int(match.group(1)) if match else None


def zarr_first_dimension(entry):
    shapes = [
        array["shape"]
        for array in entry.get("arrays", [])
        if array.get("shape")
    ]

    if not shapes:
        raise ValueError(f"No arrays found in Zarr store: {entry['path']}")

    first_dims = {int(shape[0]) for shape in shapes}

    if len(first_dims) != 1:
        raise ValueError(
            f"Inconsistent first dimensions in {entry['path']}: "
            f"{sorted(first_dims)}"
        )

    return first_dims.pop()


# ------------------------------------------------------------
# 1. Find the yearly MODIS and Sentinel-1 Zarr stores.
# ------------------------------------------------------------

modis_store_info = {}
s1_store_info = {}

for entry in zarr_inventory:
    if entry.get("error") or not entry.get("path"):
        continue

    path_text = entry["path"].replace("\\", "/").lower()
    year = zarr_year_from_entry(entry)

    if year not in YEARS or not entry.get("arrays"):
        continue

    info = {
        "path": entry["path"],
        "rows": zarr_first_dimension(entry),
    }

    if (
        "evi_lst_patch_dataset/" in path_text
        and path_text.endswith("/patches.zarr")
    ):
        if year in modis_store_info:
            raise ValueError(f"Multiple MODIS patch stores found for {year}.")
        modis_store_info[year] = info

    elif (
        "nearest_valid_overview_v1" in path_text
        and "s1_nearest_valid_" in path_text
    ):
        if year in s1_store_info:
            raise ValueError(f"Multiple Sentinel-1 stores found for {year}.")
        s1_store_info[year] = info


missing_modis_years = sorted(set(YEARS) - set(modis_store_info))
missing_s1_years = sorted(set(YEARS) - set(s1_store_info))

if missing_modis_years or missing_s1_years:
    raise FileNotFoundError(
        f"Missing yearly Zarr stores. "
        f"MODIS={missing_modis_years}; Sentinel-1={missing_s1_years}"
    )

print("Yearly Zarr stores located:")
for year in YEARS:
    print(
        f"  {year}: MODIS={modis_store_info[year]['rows']:,} rows; "
        f"Sentinel-1={s1_store_info[year]['rows']:,} rows"
    )


# Remove old views in dependency order so this cell can be rerun safely.
for view_name in [
    "s1_index",
    "s1_parquet_index",
    "s1_store_info",
    "modis_zarr_index",
]:
    con.execute(f"DROP VIEW IF EXISTS {view_name}")


# ------------------------------------------------------------
# 2. Build the MODIS index from each samples.parquet sidecar.
#
# IMPORTANT:
# row_index is the physical record position in samples.parquet.
# This assumes the original writer used the same order when
# writing the patches to patches.zarr. Check the MODIS builder
# if that ordering has not yet been established.
# ------------------------------------------------------------

modis_index_files = []

for year in YEARS:
    store_path = ROOT / modis_store_info[year]["path"]
    sidecar_path = store_path.parent / "samples.parquet"
    zarr_rows = modis_store_info[year]["rows"]

    if not sidecar_path.exists():
        raise FileNotFoundError(f"Missing MODIS sidecar: {sidecar_path}")

    parquet_file = pq.ParquetFile(sidecar_path)

    required_columns = {
        "sample_id", "station_id", "sample_date", "modis_tile"
    }
    missing_columns = required_columns - set(
        parquet_file.schema_arrow.names
    )

    if missing_columns:
        raise ValueError(
            f"{sidecar_path} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    sidecar_rows = parquet_file.metadata.num_rows

    if sidecar_rows != zarr_rows:
        raise ValueError(
            f"MODIS {year}: sidecar has {sidecar_rows:,} rows, "
            f"but Zarr arrays have {zarr_rows:,}. Refusing to align."
        )

    index_path = (
        INDEX_DIR / f"modis_samples_row_index_{year}.parquet"
    )

    if index_path.exists():
        index_path.unlink()

    writer = None
    offset = 0

    try:
        for batch in parquet_file.iter_batches(
            columns=[
                "sample_id",
                "station_id",
                "sample_date",
                "modis_tile",
            ],
            batch_size=INDEX_BATCH_ROWS,
        ):
            n = batch.num_rows

            table = pa.table({
                "sample_id": batch.column(
                    batch.schema.get_field_index("sample_id")
                ).cast(pa.int64()),
                "station_id": batch.column(
                    batch.schema.get_field_index("station_id")
                ).cast(pa.string()),
                "sample_date": batch.column(
                    batch.schema.get_field_index("sample_date")
                ).cast(pa.date32()),
                "modis_tile": batch.column(
                    batch.schema.get_field_index("modis_tile")
                ).cast(pa.string()),
                "modis_index_year": pa.array(
                    np.full(n, year, dtype=np.int32)
                ),
                "modis_patch_row_index": pa.array(
                    np.arange(offset, offset + n, dtype=np.int64)
                ),
                "modis_index_store_path": pa.array(
                    [modis_store_info[year]["path"]] * n,
                    type=pa.string(),
                ),
            })

            if writer is None:
                writer = pq.ParquetWriter(
                    index_path,
                    table.schema,
                    compression="zstd",
                )

            writer.write_table(table)
            offset += n

    finally:
        if writer is not None:
            writer.close()

    if offset != zarr_rows:
        raise ValueError(
            f"MODIS {year}: indexed {offset:,} rows, "
            f"expected {zarr_rows:,}."
        )

    modis_index_files.append(index_path)
    print(f"MODIS {year}: indexed {offset:,} rows")


if not modis_index_files:
    raise ValueError("No MODIS sample indexes were created.")

modis_paths_sql = ", ".join(
    sqlq(path.resolve()) for path in modis_index_files
)

con.execute(f"""
CREATE OR REPLACE VIEW modis_zarr_index AS
SELECT
    CAST(sample_id AS BIGINT) AS sample_id,
    TRIM(CAST(station_id AS VARCHAR)) AS station_id,
    CAST(sample_date AS DATE) AS sample_date,
    CAST(modis_index_year AS INTEGER) AS modis_index_year,
    CAST(modis_patch_row_index AS BIGINT) AS modis_patch_row_index,
    CAST(modis_tile AS VARCHAR) AS modis_tile,
    CAST(modis_index_store_path AS VARCHAR) AS modis_index_store_path
FROM read_parquet([{modis_paths_sql}], union_by_name=true)
""")


# Validate MODIS index uniqueness and identity against canonical samples.
modis_duplicates = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT sample_id
    FROM modis_zarr_index
    GROUP BY sample_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

modis_missing = con.execute(f"""
SELECT COUNT(*)
FROM read_parquet({sqlq(CANDIDATES.resolve())}) b
LEFT JOIN modis_zarr_index m USING (sample_id)
WHERE m.sample_id IS NULL
""").fetchone()[0]

modis_extra = con.execute(f"""
SELECT COUNT(*)
FROM modis_zarr_index m
LEFT JOIN read_parquet({sqlq(CANDIDATES.resolve())}) b USING (sample_id)
WHERE b.sample_id IS NULL
""").fetchone()[0]

modis_identity_mismatches = con.execute(f"""
SELECT COUNT(*)
FROM modis_zarr_index m
JOIN read_parquet({sqlq(CANDIDATES.resolve())}) b USING (sample_id)
WHERE TRIM(m.station_id) IS DISTINCT FROM
          TRIM(CAST(b.station_id AS VARCHAR))
   OR m.sample_date IS DISTINCT FROM CAST(b.sample_date AS DATE)
   OR m.modis_index_year IS DISTINCT FROM
          EXTRACT(YEAR FROM CAST(b.sample_date AS DATE))::INTEGER
""").fetchone()[0]

print("\nMODIS index validation:")
print("  Duplicate sample IDs:", f"{modis_duplicates:,}")
print("  Canonical samples without mapping:", f"{modis_missing:,}")
print("  Index IDs absent from canonical samples:", f"{modis_extra:,}")
print("  Station/date/year mismatches:", f"{modis_identity_mismatches:,}")

if any([
    modis_duplicates,
    modis_missing,
    modis_extra,
    modis_identity_mismatches,
]):
    raise ValueError(
        "MODIS index identity validation failed. "
        "Do not install the merged outputs."
    )


# ------------------------------------------------------------
# 3. Build the Sentinel-1 index from the correct yearly files.
# ------------------------------------------------------------

s1_index_paths = [
    S1_ROOT / f"samples_nearest_valid_{year}.parquet"
    for year in YEARS
]

missing_s1_files = [
    path for path in s1_index_paths if not path.exists()
]

if missing_s1_files:
    raise FileNotFoundError(
        "Missing Sentinel-1 index files:\n"
        + "\n".join(str(path) for path in missing_s1_files)
    )

s1_paths_sql = ", ".join(
    sqlq(path.resolve()) for path in s1_index_paths
)

s1_store_values = ", ".join(
    "("
    f"{year}, "
    f"{sqlq(s1_store_info[year]['path'])}, "
    f"{s1_store_info[year]['rows']}"
    ")"
    for year in YEARS
)

con.execute(f"""
CREATE OR REPLACE TEMP VIEW s1_store_info AS
SELECT
    CAST(year AS INTEGER) AS year,
    CAST(store_path AS VARCHAR) AS store_path,
    CAST(zarr_rows AS BIGINT) AS zarr_rows
FROM (
    VALUES {s1_store_values}
) AS v(year, store_path, zarr_rows)
""")

con.execute(f"""
CREATE OR REPLACE VIEW s1_parquet_index AS
SELECT
    CAST(sample_id AS BIGINT) AS sample_id,
    TRIM(CAST(station_id AS VARCHAR)) AS station_id,
    CAST(sample_date AS DATE) AS sample_date,
    CAST(year AS INTEGER) AS s1_index_year,
    CAST(row_index AS BIGINT) AS s1_patch_row_index
FROM read_parquet([{s1_paths_sql}], union_by_name=true)
""")

s1_year_stats = con.execute("""
SELECT
    p.s1_index_year,
    COUNT(*) AS index_rows,
    COUNT(DISTINCT p.s1_patch_row_index) AS unique_row_indices,
    MIN(p.s1_patch_row_index) AS min_row_index,
    MAX(p.s1_patch_row_index) AS max_row_index,
    MAX(i.zarr_rows) AS zarr_rows
FROM s1_parquet_index p
JOIN s1_store_info i ON i.year = p.s1_index_year
GROUP BY p.s1_index_year
ORDER BY p.s1_index_year
""").fetchdf()

print("\nSentinel-1 yearly index validation:")
display(s1_year_stats)

for _, row in s1_year_stats.iterrows():
    year = int(row["s1_index_year"])
    n = int(row["index_rows"])

    if (
        n != int(row["zarr_rows"])
        or int(row["unique_row_indices"]) != n
        or int(row["min_row_index"]) != 0
        or int(row["max_row_index"]) != n - 1
    ):
        raise ValueError(
            f"Sentinel-1 {year}: row-index range does not exactly "
            f"match the Zarr array length."
        )

if set(s1_year_stats["s1_index_year"].astype(int)) != set(YEARS):
    raise ValueError("One or more Sentinel-1 years are absent from the index.")


s1_duplicates = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT sample_id
    FROM s1_parquet_index
    GROUP BY sample_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

s1_missing = con.execute(f"""
SELECT COUNT(*)
FROM read_parquet({sqlq(CANDIDATES.resolve())}) b
LEFT JOIN s1_parquet_index s USING (sample_id)
WHERE s.sample_id IS NULL
""").fetchone()[0]

s1_extra = con.execute(f"""
SELECT COUNT(*)
FROM s1_parquet_index s
LEFT JOIN read_parquet({sqlq(CANDIDATES.resolve())}) b USING (sample_id)
WHERE b.sample_id IS NULL
""").fetchone()[0]

s1_identity_mismatches = con.execute(f"""
SELECT COUNT(*)
FROM s1_parquet_index s
JOIN read_parquet({sqlq(CANDIDATES.resolve())}) b USING (sample_id)
WHERE TRIM(s.station_id) IS DISTINCT FROM
          TRIM(CAST(b.station_id AS VARCHAR))
   OR s.sample_date IS DISTINCT FROM CAST(b.sample_date AS DATE)
   OR s.s1_index_year IS DISTINCT FROM
          EXTRACT(YEAR FROM CAST(b.sample_date AS DATE))::INTEGER
""").fetchone()[0]

print("\nSentinel-1 index identity validation:")
print("  Duplicate sample IDs:", f"{s1_duplicates:,}")
print("  Canonical samples without mapping:", f"{s1_missing:,}")
print("  Index IDs absent from canonical samples:", f"{s1_extra:,}")
print("  Station/date/year mismatches:", f"{s1_identity_mismatches:,}")

if any([
    s1_duplicates,
    s1_missing,
    s1_extra,
    s1_identity_mismatches,
]):
    raise ValueError(
        "Sentinel-1 index identity validation failed. "
        "Do not install the merged outputs."
    )

con.execute("""
CREATE OR REPLACE VIEW s1_index AS
SELECT
    p.sample_id,
    p.s1_index_year,
    p.s1_patch_row_index,
    i.store_path AS s1_index_store_path
FROM s1_parquet_index p
JOIN s1_store_info i ON i.year = p.s1_index_year
""")


# ------------------------------------------------------------
# 4. Validate one-row-per-station cluster metadata.
# ------------------------------------------------------------

cluster_dup = con.execute(f"""
SELECT COUNT(*)
FROM (
    SELECT network, station_id
    FROM read_parquet({sqlq(S1_STATION_CLUSTERS.resolve())})
    GROUP BY network, station_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

if cluster_dup:
    raise ValueError(
        "stations_with_clusters.parquet has duplicate network/station rows."
    )


# ------------------------------------------------------------
# 5. Build the scalar feature table.
# ------------------------------------------------------------

FEATURES_TMP = WORK_DIR / "merged_sample_features.parquet"

feature_sql = f"""
COPY (
 SELECT
   b.*,
   e.era5_point_id, e.era5_lat, e.era5_lon,
   e.t2m_K, e.d2m_K, e.precip_mm, e.VPD_kPa, e.API_mm,
   sm.station_key, sm.SMAP, sm.SMAP_n_obs, sm.SMAP_cell_conflict,
   soil.clay_pct AS static_clay_pct,
   soil.sand_pct AS static_sand_pct,
   soil.silt_pct AS static_silt_pct,
   terrain.slope_deg, terrain.aspect_deg,
   terrain.elevation_m AS static_elevation_m,
   sc.cluster_id AS sentinel1_cluster_id,
   si.s1_index_year,
   si.s1_patch_row_index,
   si.s1_index_store_path,
   mi.modis_index_year,
   mi.modis_patch_row_index,
   mi.modis_index_store_path,
   mi.modis_tile
 FROM read_parquet({sqlq(CANDIDATES.resolve())}) b
 LEFT JOIN read_parquet({sqlq(ERA5.resolve())}) e USING(sample_id)
 LEFT JOIN read_parquet({sqlq(SMAP.resolve())}) sm USING(sample_id)
 LEFT JOIN static_soil soil
   ON soil.station_key =
      TRIM(CAST(b.network AS VARCHAR)) || '::' ||
      TRIM(CAST(b.station_id AS VARCHAR))
 LEFT JOIN static_terrain terrain
   ON terrain.station_key =
      TRIM(CAST(b.network AS VARCHAR)) || '::' ||
      TRIM(CAST(b.station_id AS VARCHAR))
 LEFT JOIN read_parquet({sqlq(S1_STATION_CLUSTERS.resolve())}) sc
   ON TRIM(CAST(sc.network AS VARCHAR)) =
      TRIM(CAST(b.network AS VARCHAR))
  AND TRIM(CAST(sc.station_id AS VARCHAR)) =
      TRIM(CAST(b.station_id AS VARCHAR))
 LEFT JOIN s1_index si USING(sample_id)
 LEFT JOIN modis_zarr_index mi USING(sample_id)
) TO {sqlq(FEATURES_TMP.resolve())}
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
"""

con.execute(feature_sql)

print(
    "Temporary feature table:",
    FEATURES_TMP,
    f"({FEATURES_TMP.stat().st_size:,} bytes)"
)
print("Cell 9 completed successfully.")

Yearly Zarr stores located:
  2015: MODIS=329,903 rows; Sentinel-1=329,903 rows
  2016: MODIS=394,790 rows; Sentinel-1=394,790 rows
  2017: MODIS=369,953 rows; Sentinel-1=369,953 rows
  2018: MODIS=344,806 rows; Sentinel-1=344,806 rows
  2019: MODIS=308,679 rows; Sentinel-1=308,679 rows
  2020: MODIS=269,853 rows; Sentinel-1=269,853 rows
  2021: MODIS=259,625 rows; Sentinel-1=259,625 rows
  2022: MODIS=260,490 rows; Sentinel-1=260,490 rows
  2023: MODIS=233,307 rows; Sentinel-1=233,307 rows
  2024: MODIS=264,946 rows; Sentinel-1=264,946 rows
  2025: MODIS=270,736 rows; Sentinel-1=270,736 rows
  2026: MODIS=207,555 rows; Sentinel-1=207,555 rows
MODIS 2015: indexed 329,903 rows
MODIS 2016: indexed 394,790 rows
MODIS 2017: indexed 369,953 rows
MODIS 2018: indexed 344,806 rows
MODIS 2019: indexed 308,679 rows
MODIS 2020: indexed 269,853 rows
MODIS 2021: indexed 259,625 rows
MODIS 2022: indexed 260,490 rows
MODIS 2023: indexed 233,307 rows
MODIS 2024: indexed 264,946 rows
MODIS 2025: indexe

,s1_index_year,index_rows,unique_row_indices,min_row_index,max_row_index,zarr_rows
0,2015,329903,329903,0,329902,329903
1,2016,394790,394790,0,394789,394790
2,2017,369953,369953,0,369952,369953
3,2018,344806,344806,0,344805,344806
4,2019,308679,308679,0,308678,308679
5,2020,269853,269853,0,269852,269853
6,2021,259625,259625,0,259624,259625
7,2022,260490,260490,0,260489,260490
8,2023,233307,233307,0,233306,233307
9,2024,264946,264946,0,264945,264946



Sentinel-1 index identity validation:
  Duplicate sample IDs: 0
  Canonical samples without mapping: 0
  Index IDs absent from canonical samples: 0
  Station/date/year mismatches: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Temporary feature table: /home/mayur/Code/UG_SM/Data/Global/.global_merge_work_htk4igmm/merged_sample_features.parquet (92,416,604 bytes)
Cell 9 completed successfully.


In [7]:
# Validate feature count, ID uniqueness, and broad feature coverage.
feature_qc = con.execute(f"""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT sample_id) AS distinct_sample_ids,
       SUM(CASE WHEN t2m_K IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_era5,
       SUM(CASE WHEN SMAP IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_smap,
       SUM(CASE WHEN static_clay_pct IS NOT NULL OR static_sand_pct IS NOT NULL OR static_silt_pct IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_static_soil,
       SUM(CASE WHEN slope_deg IS NOT NULL OR aspect_deg IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_static_terrain,
       SUM(CASE WHEN sentinel1_cluster_id IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_sentinel_cluster,
       SUM(CASE WHEN s1_patch_row_index IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_explicit_s1_row_index,
       SUM(CASE WHEN modis_patch_row_index IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_explicit_modis_row_index
FROM read_parquet({sqlq(FEATURES_TMP.resolve())})
""").fetchdf()
display(feature_qc)
out_rows = int(feature_qc.loc[0, "row_count"])
out_ids = int(feature_qc.loc[0, "distinct_sample_ids"])
if out_rows != base_rows or out_ids != base_ids:
    raise ValueError("Feature join changed canonical row count or sample IDs; do not install output.")
print("Feature output row-count validation passed.")


,row_count,distinct_sample_ids,rows_with_era5,rows_with_smap,rows_with_static_soil,rows_with_static_terrain,rows_with_sentinel_cluster,rows_with_explicit_s1_row_index,rows_with_explicit_modis_row_index
0,3514643,3514643,3489385.0,3510848.0,3436272.0,3514643.0,3514643.0,3514643.0,3514643.0


Feature output row-count validation passed.


## 4. Attach `sample_id` to every daily ISMN target observation

The daily table is larger than the canonical sample index because it preserves individual measurement records across sensors and depths. This produces a long-format target table and keeps all source rows. Records with no matching candidate sample are retained with a null `sample_id` and counted in validation.

In [8]:
TARGETS_TMP = WORK_DIR / "global_target_observations.parquet"
target_sql = f"""
COPY (
 SELECT
   b.sample_id,
   d.station_id, d.network, d.network_name, d.station,
   CAST(d.sample_date AS DATE) AS sample_date,
   d.latitude, d.longitude, d.elevation_m,
   d.sensor, d.depth_from_m, d.depth_to_m, d.depth_class,
   d.soil_moisture_m3_m3, d.observation_count, d.source_file,
   d.clay_pct AS ismn_source_clay_pct,
   d.sand_pct AS ismn_source_sand_pct,
   d.silt_pct AS ismn_source_silt_pct,
   d.climate_zone
 FROM read_parquet({sqlq(DAILY.resolve())}) d
 LEFT JOIN read_parquet({sqlq(CANDIDATES.resolve())}) b
   ON TRIM(CAST(b.network AS VARCHAR)) = TRIM(CAST(d.network AS VARCHAR))
  AND TRIM(CAST(b.station_id AS VARCHAR)) = TRIM(CAST(d.station_id AS VARCHAR))
  AND CAST(b.sample_date AS DATE) = CAST(d.sample_date AS DATE)
) TO {sqlq(TARGETS_TMP.resolve())}
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
"""
con.execute(target_sql)
label_qc = con.execute(f"""
SELECT COUNT(*) AS target_observation_rows,
       COUNT(sample_id) AS rows_linked_to_sample,
       SUM(CASE WHEN sample_id IS NULL THEN 1 ELSE 0 END) AS rows_without_sample_id,
       COUNT(DISTINCT sample_id) AS distinct_linked_samples,
       SUM(CASE WHEN soil_moisture_m3_m3 IS NOT NULL THEN 1 ELSE 0 END) AS nonnull_moisture_labels
FROM read_parquet({sqlq(TARGETS_TMP.resolve())})
""").fetchdf()
display(label_qc)
print("Temporary target table:", TARGETS_TMP, f"({TARGETS_TMP.stat().st_size:,} bytes)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,target_observation_rows,rows_linked_to_sample,rows_without_sample_id,distinct_linked_samples,nonnull_moisture_labels
0,5045849,5045849,0.0,3514643,5045849.0


Temporary target table: /home/mayur/Code/UG_SM/Data/Global/.global_merge_work_htk4igmm/global_target_observations.parquet (25,986,072 bytes)


In [9]:
# ============================================================
# CELL 13: BUILD SPATIAL ALIGNMENT MANIFEST
# ============================================================

# Store paths are pointers to Zarr datasets. Row indices come from
# the validated sidecar/index views created in Cell 9.

modis_store_by_year = {
    year: info["path"]
    for year, info in modis_store_info.items()
}

s1_store_by_year = {
    year: info["path"]
    for year, info in s1_store_info.items()
}

modis_values = ", ".join(
    f"({year}, {sqlq(path)})"
    for year, path in sorted(modis_store_by_year.items())
)

s1_values = ", ".join(
    f"({year}, {sqlq(path)})"
    for year, path in sorted(s1_store_by_year.items())
)

SPATIAL_TMP = WORK_DIR / "spatial_alignment_manifest.parquet"

spatial_sql = f"""
COPY (
 SELECT
   b.sample_id,
   CAST(b.sample_date AS DATE) AS sample_date,
   EXTRACT(
       YEAR FROM CAST(b.sample_date AS DATE)
   )::INTEGER AS sample_year,

   COALESCE(mi.modis_index_store_path, mp.store_path)
       AS modis_zarr_path,
   mi.modis_index_year,
   mi.modis_patch_row_index,
   mi.modis_tile,

   CASE
     WHEN mi.sample_id IS NOT NULL
       THEN 'mapped_by_samples_parquet_row_order_assumption'
     ELSE 'row_mapping_not_found; do_not_assume_order'
   END AS modis_alignment_status,

   COALESCE(si.s1_index_store_path, sp.store_path)
       AS sentinel1_zarr_path,
   si.s1_index_year,
   si.s1_patch_row_index,

   CASE
     WHEN si.sample_id IS NOT NULL
       THEN 'mapped_by_year_specific_sample_index'
     ELSE 'row_mapping_not_found; do_not_assume_order'
   END AS sentinel1_alignment_status

 FROM read_parquet({sqlq(CANDIDATES.resolve())}) b

 LEFT JOIN (
     SELECT *
     FROM (VALUES {modis_values}) AS t(year, store_path)
 ) mp
   ON mp.year =
      EXTRACT(YEAR FROM CAST(b.sample_date AS DATE))

 LEFT JOIN (
     SELECT *
     FROM (VALUES {s1_values}) AS t(year, store_path)
 ) sp
   ON sp.year =
      EXTRACT(YEAR FROM CAST(b.sample_date AS DATE))

 LEFT JOIN modis_zarr_index mi USING(sample_id)
 LEFT JOIN s1_index si USING(sample_id)

) TO {sqlq(SPATIAL_TMP.resolve())}
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
"""

con.execute(spatial_sql)


spatial_qc = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT sample_id) AS distinct_sample_ids,

    COUNT(*) FILTER (
        WHERE modis_patch_row_index IS NOT NULL
    ) AS rows_with_modis_row_index,

    COUNT(*) FILTER (
        WHERE s1_patch_row_index IS NOT NULL
    ) AS rows_with_sentinel1_row_index,

    COUNT(*) FILTER (
        WHERE modis_patch_row_index IS NULL
    ) AS modis_rows_missing,

    COUNT(*) FILTER (
        WHERE s1_patch_row_index IS NULL
    ) AS sentinel1_rows_missing,

    COUNT(*) FILTER (
        WHERE modis_patch_row_index IS NOT NULL
          AND modis_zarr_path IS NULL
    ) AS modis_rows_missing_path,

    COUNT(*) FILTER (
        WHERE s1_patch_row_index IS NOT NULL
          AND sentinel1_zarr_path IS NULL
    ) AS sentinel1_rows_missing_path

FROM read_parquet({sqlq(SPATIAL_TMP.resolve())})
""").fetchdf()

display(spatial_qc)


spatial_rows = int(spatial_qc.loc[0, "row_count"])
spatial_ids = int(spatial_qc.loc[0, "distinct_sample_ids"])

if spatial_rows != base_rows or spatial_ids != base_ids:
    raise ValueError(
        "Spatial manifest changed the canonical row count or sample IDs."
    )

modis_indexed = int(
    spatial_qc.loc[0, "rows_with_modis_row_index"]
)
s1_indexed = int(
    spatial_qc.loc[0, "rows_with_sentinel1_row_index"]
)

print("\nSpatial manifest validation:")
print(f"  Canonical samples: {base_rows:,}")
print(f"  MODIS indexed: {modis_indexed:,}")
print(f"  Sentinel-1 indexed: {s1_indexed:,}")

if modis_indexed != base_rows:
    print(
        "WARNING: MODIS row-index coverage is incomplete. "
        "Do not assume missing rows correspond to valid patches."
    )

if s1_indexed != base_rows:
    print(
        "WARNING: Sentinel-1 row-index coverage is incomplete. "
        "Inspect the year-specific index files."
    )

print("Cell 13 completed successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,distinct_sample_ids,rows_with_modis_row_index,rows_with_sentinel1_row_index,modis_rows_missing,sentinel1_rows_missing,modis_rows_missing_path,sentinel1_rows_missing_path
0,3514643,3514643,3514643,3514643,0,0,0,0



Spatial manifest validation:
  Canonical samples: 3,514,643
  MODIS indexed: 3,514,643
  Sentinel-1 indexed: 3,514,643
Cell 13 completed successfully.


## 5. Final validation and safe installation

The outputs are first written to a temporary work directory. This cell validates the sample-level row counts, writes a JSON report, backs up any existing output, and installs the validated files. Existing Zarr arrays are not modified.

In [10]:
feature_summary = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT sample_id) AS distinct_sample_ids,
    SUM(CASE WHEN SMAP_cell_conflict THEN 1 ELSE 0 END) AS smap_conflict_rows,
    SUM(CASE WHEN SMAP IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_smap,
    SUM(CASE WHEN t2m_K IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_era5
FROM read_parquet({sqlq(FEATURES_TMP.resolve())})
""").fetchone()

target_summary = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    SUM(CASE WHEN sample_id IS NULL THEN 1 ELSE 0 END) AS unlinked_label_rows,
    COUNT(DISTINCT sample_id) AS linked_sample_ids
FROM read_parquet({sqlq(TARGETS_TMP.resolve())})
""").fetchone()

spatial_summary = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT sample_id) AS distinct_sample_ids,
    SUM(CASE WHEN modis_patch_row_index IS NOT NULL THEN 1 ELSE 0 END) AS modis_rows_indexed,
    SUM(CASE WHEN s1_patch_row_index IS NOT NULL THEN 1 ELSE 0 END) AS s1_rows_indexed
FROM read_parquet({sqlq(SPATIAL_TMP.resolve())})
""").fetchone()

validation = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "root": str(ROOT.resolve()),
    "canonical": {
        "rows": int(base_rows),
        "distinct_sample_ids": int(base_ids),
    },
    "source_scalar_checks": source_qc,
    "feature_output": dict(zip(
        ["rows", "distinct_sample_ids", "smap_conflict_rows",
         "rows_with_smap", "rows_with_era5"],
        [None if x is None else int(x) for x in feature_summary],
    )),
    "target_observations_output": dict(zip(
        ["rows", "unlinked_label_rows", "linked_sample_ids"],
        [None if x is None else int(x) for x in target_summary],
    )),
    "spatial_manifest_output": dict(zip(
        ["rows", "distinct_sample_ids", "modis_rows_indexed",
         "s1_rows_indexed"],
        [None if x is None else int(x) for x in spatial_summary],
    )),
    "zarr_inventory": inventory_for_report,
    "outputs": [str(FEATURES_OUT), str(TARGETS_OUT), str(SPATIAL_OUT)],
    "albedo": "skipped by design",
    "target_semantics": (
        "Daily ISMN records preserved per sensor/depth; "
        "no averaging rule applied."
    ),
}

VALIDATION_TMP = WORK_DIR / "global_merge_validation.json"
VALIDATION_TMP.write_text(
    json.dumps(validation, indent=2, default=str),
    encoding="utf-8",
)

print(json.dumps({
    k: validation[k]
    for k in [
        "canonical",
        "source_scalar_checks",
        "feature_output",
        "target_observations_output",
        "spatial_manifest_output",
    ]
}, indent=2))

if int(feature_summary[0]) != base_rows or int(feature_summary[1]) != base_ids:
    raise ValueError(
        "Feature table did not preserve sample count/IDs; target files are not installed."
    )

if int(target_summary[0]) != parquet_rows(DAILY):
    raise ValueError(
        "Target observation table did not preserve the original ISMN daily row count."
    )

if int(spatial_summary[0]) != base_rows or int(spatial_summary[1]) != base_ids:
    raise ValueError(
        "Spatial manifest did not preserve sample count/IDs."
    )

for src, dst in [
    (FEATURES_TMP, FEATURES_OUT),
    (TARGETS_TMP, TARGETS_OUT),
    (SPATIAL_TMP, SPATIAL_OUT),
    (VALIDATION_TMP, VALIDATION_OUT),
]:
    if dst.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = dst.with_name(
            f"{dst.stem}.backup_{stamp}{dst.suffix}"
        )
        shutil.copy2(dst, backup)
        print("Backup:", backup)

    src.replace(dst)

print("\nInstalled outputs:")
for p in [
    FEATURES_OUT,
    TARGETS_OUT,
    SPATIAL_OUT,
    VALIDATION_OUT,
    ROOT / "global_zarr_inventory.json",
]:
    if p.exists():
        print(f"  {p.name}: {p.stat().st_size:,} bytes")

con.close()
shutil.rmtree(WORK_DIR, ignore_errors=True)
print("Temporary work directory cleaned.")

{
  "canonical": {
    "rows": 3514643,
    "distinct_sample_ids": 3514643
  },
  "source_scalar_checks": {
    "ERA5": {
      "rows": 3514643,
      "distinct_sample_ids": 3514643,
      "duplicate_sample_id_groups": 0,
      "candidate_ids_missing_in_source": 0,
      "source_ids_not_in_candidates": 0,
      "sample_id_station_date_mismatches": 0
    },
    "SMAP": {
      "rows": 3514643,
      "distinct_sample_ids": 3514643,
      "duplicate_sample_id_groups": 0,
      "candidate_ids_missing_in_source": 0,
      "source_ids_not_in_candidates": 0,
      "sample_id_station_date_mismatches": 0
    }
  },
  "feature_output": {
    "rows": 3514643,
    "distinct_sample_ids": 3514643,
    "smap_conflict_rows": 457318,
    "rows_with_smap": 3510848,
    "rows_with_era5": 3489385
  },
  "target_observations_output": {
    "rows": 5045849,
    "unlinked_label_rows": 0,
    "linked_sample_ids": 3514643
  },
  "spatial_manifest_output": {
    "rows": 3514643,
    "distinct_sample_ids": 35146

## Important interpretation notes

1. `global_target_observations.parquet` preserves individual labels by sensor and depth. Before training a single-target model, choose an explicit depth/sensor policy or a documented aggregation rule.
2. `SMAP_cell_conflict` is retained in `merged_sample_features.parquet` for filtering or analysis.
3. In `spatial_alignment_manifest.parquet`, Sentinel-1 row indices come from explicit year-specific index Parquets and are checked against the Zarr row counts. MODIS row indices are generated from the order of `samples.parquet`; this is only an assumption until the MODIS extraction writer is confirmed to have written patches in exactly that same order. A matching row count alone does not prove alignment.
4. `global_zarr_inventory.json` lists the Zarr arrays. If MODIS sample IDs are not embedded in the store, the MODIS patch writer or an authoritative sample-to-row index must be inspected to confirm row order before using the generated MODIS patch indices as verified mappings.
5. The schema report said `modis_patch_dataset` and `ndvi_lst_patch_dataset` were not found at the listed paths. This notebook reports those paths independently; check `global_zarr_inventory.json` if the directories have moved.
6. Raster values remain in Zarr and are accessed lazily by the model using the spatial manifest; they are not copied into a huge flat Parquet.

In [11]:
import json
from pathlib import Path

root = globals().get("ROOT", Path.home() / "Code/UG_SM/Data/Global")
report_path = root / "global_merge_validation.json"

if not report_path.exists():
    print("Validation report not found. The final merge cell may not have completed.")
else:
    report = json.loads(report_path.read_text())

    print("Canonical samples:", report["canonical"])
    print("\nFeature output:")
    print(json.dumps(report["feature_output"], indent=2))

    print("\nTarget observations:")
    print(json.dumps(report["target_observations_output"], indent=2))

    print("\nSpatial alignment:")
    print(json.dumps(report["spatial_manifest_output"], indent=2))

    print("\nZarr inventory:")
    for store in report.get("zarr_inventory", []):
        print(
            store.get("path", store.get("root", "unknown")),
            "| arrays:", len(store.get("arrays", [])),
            "| sample-ID mapping:", store.get("sample_id_index_extracted", False),
            "| error:", store.get("error", "none"),
        )

Canonical samples: {'rows': 3514643, 'distinct_sample_ids': 3514643}

Feature output:
{
  "rows": 3514643,
  "distinct_sample_ids": 3514643,
  "smap_conflict_rows": 457318,
  "rows_with_smap": 3510848,
  "rows_with_era5": 3489385
}

Target observations:
{
  "rows": 5045849,
  "unlinked_label_rows": 0,
  "linked_sample_ids": 3514643
}

Spatial alignment:
{
  "rows": 3514643,
  "distinct_sample_ids": 3514643,
  "modis_rows_indexed": 3514643,
  "s1_rows_indexed": 3514643
}

Zarr inventory:
modis/evi_lst_patch_dataset/2015/patches.zarr | arrays: 3 | sample-ID mapping: False | error: none
modis/evi_lst_patch_dataset/2016/patches.zarr | arrays: 3 | sample-ID mapping: False | error: none
modis/evi_lst_patch_dataset/2017/patches.zarr | arrays: 3 | sample-ID mapping: False | error: none
modis/evi_lst_patch_dataset/2018/patches.zarr | arrays: 3 | sample-ID mapping: False | error: none
modis/evi_lst_patch_dataset/2019/patches.zarr | arrays: 3 | sample-ID mapping: False | error: none
modis/evi_lst

In [12]:
from pathlib import Path
import json
import duckdb
import zarr
import pyarrow.parquet as pq

ROOT = globals().get("ROOT", Path.home() / "Code/UG_SM/Data/Global")
con_check = duckdb.connect(":memory:")
con_check.execute("SET memory_limit='512MB'")
con_check.execute("SET threads=2")

def q(path):
    return "'" + str(path).replace("'", "''") + "'"

# 1. Print the actual names and shapes of arrays in each Zarr store.
inventory_path = ROOT / "global_zarr_inventory.json"

if inventory_path.exists():
    inventory = json.loads(inventory_path.read_text())
    for store in inventory:
        print("\nSTORE:", store.get("path", store.get("root")))
        for array in store.get("arrays", []):
            print(
                f"  {array['name']}: shape={array['shape']}, "
                f"dtype={array['dtype']}, chunks={array.get('chunks')}"
            )
else:
    print("global_zarr_inventory.json not found.")

# 2. Compare the Sentinel sample-index files with their Zarr row counts.
for year in range(2015, 2027):
    index_path = ROOT / "sentinel1/nearest_valid_overview_v1" / f"samples_nearest_valid_{year}.parquet"
    zarr_path = (
        ROOT / "sentinel1/nearest_valid_overview_v1"
        / f"s1_nearest_valid_{year}.zarr"
    )

    print(f"\nSENTINEL-1 {year}")

    if not index_path.exists():
        print("  Sample-index Parquet: MISSING")
    else:
        pf = pq.ParquetFile(index_path)
        print(f"  Index rows: {pf.metadata.num_rows:,}")

        stats = con_check.execute(f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(DISTINCT sample_id) AS distinct_ids,
                COUNT(DISTINCT row_index) AS distinct_row_indices,
                MIN(row_index) AS min_row_index,
                MAX(row_index) AS max_row_index
            FROM read_parquet({q(index_path)})
        """).fetchone()
        print("  Index stats:", stats)
        n, ids, idxs, lo, hi = stats
        print(
            "  Contiguous zero-based row indices:",
            bool(n and lo == 0 and hi == n - 1 and idxs == n)
        )

    if not zarr_path.exists():
        print("  Zarr store: MISSING")
        continue

    try:
        group = zarr.open_group(str(zarr_path), mode="r")
        print("  Zarr arrays:")
        try:
            arrays = group.arrays()
        except Exception:
            arrays = []

        for name, arr in arrays:
            print(
                f"    {name}: shape={arr.shape}, "
                f"dtype={arr.dtype}, chunks={getattr(arr, 'chunks', None)}"
            )
    except Exception as exc:
        print("  Could not inspect store:", repr(exc))

# 3. Look for MODIS sidecar mapping files beside the year-wise stores.
print("\nMODIS SIDECAR FILES")
modis_root = ROOT / "modis"

for dataset in sorted(p for p in modis_root.iterdir() if p.is_dir()):
    print("\n", dataset.relative_to(ROOT))
    for level in [dataset, *sorted(p for p in dataset.iterdir() if p.is_dir())]:
        for file in sorted(level.iterdir()):
            if file.is_file() and file.suffix.lower() in {
                ".parquet", ".csv", ".json", ".txt"
            }:
                print("  ", file.relative_to(ROOT))

con_check.close()


STORE: modis/evi_lst_patch_dataset/2015/patches.zarr
  evi: shape=[329903, 27, 27], dtype=int16, chunks=[128, 27, 27]
  done: shape=[329903], dtype=bool, chunks=[4096]
  lst: shape=[329903, 13, 13], dtype=uint16, chunks=[256, 13, 13]

STORE: modis/evi_lst_patch_dataset/2016/patches.zarr
  done: shape=[394790], dtype=bool, chunks=[4096]
  evi: shape=[394790, 27, 27], dtype=int16, chunks=[128, 27, 27]
  lst: shape=[394790, 13, 13], dtype=uint16, chunks=[256, 13, 13]

STORE: modis/evi_lst_patch_dataset/2017/patches.zarr
  evi: shape=[369953, 27, 27], dtype=int16, chunks=[128, 27, 27]
  lst: shape=[369953, 13, 13], dtype=uint16, chunks=[256, 13, 13]
  done: shape=[369953], dtype=bool, chunks=[4096]

STORE: modis/evi_lst_patch_dataset/2018/patches.zarr
  lst: shape=[344806, 13, 13], dtype=uint16, chunks=[256, 13, 13]
  evi: shape=[344806, 27, 27], dtype=int16, chunks=[128, 27, 27]
  done: shape=[344806], dtype=bool, chunks=[4096]

STORE: modis/evi_lst_patch_dataset/2019/patches.zarr
  done

/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


  Index stats: (369953, 369953, 369953, 0, 369952)
  Contiguous zero-based row indices: True
  Zarr arrays:
    hv: shape=(369953, 13, 13), dtype=float32, chunks=(256, 13, 13)
    vv_vh_ratio: shape=(369953, 13, 13), dtype=float32, chunks=(256, 13, 13)
    obs_count: shape=(369953, 4), dtype=uint8, chunks=(1024, 4)
    source_day_offset: shape=(369953, 2), dtype=int16, chunks=(1024, 2)
    vv: shape=(369953, 13, 13), dtype=float32, chunks=(256, 13, 13)
    vh: shape=(369953, 13, 13), dtype=float32, chunks=(256, 13, 13)
    sample_done: shape=(369953,), dtype=bool, chunks=(4096,)
    hh: shape=(369953, 13, 13), dtype=float32, chunks=(256, 13, 13)

SENTINEL-1 2018
  Index rows: 344,806
  Index stats: (344806, 344806, 344806, 0, 344805)
  Contiguous zero-based row indices: True
  Zarr arrays:
    hv: shape=(344806, 13, 13), dtype=float32, chunks=(256, 13, 13)
    vv: shape=(344806, 13, 13), dtype=float32, chunks=(256, 13, 13)
    vv_vh_ratio: shape=(344806, 13, 13), dtype=float32, chunks=

In [13]:
from pathlib import Path
import duckdb
import pyarrow.parquet as pq
import zarr

ROOT = globals().get("ROOT", Path.home() / "Code/UG_SM/Data/Global")

con_diag = duckdb.connect(":memory:")
con_diag.execute("SET memory_limit='512MB'")
con_diag.execute("SET threads=2")

def sqlq(value):
    return "'" + str(value).replace("'", "''") + "'"

def qid(value):
    return '"' + str(value).replace('"', '""') + '"'

def inspect_mapping_parquet(path):
    print(f"\n{'=' * 80}\nFILE: {path.relative_to(ROOT)}")
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    cols = [field.name for field in schema]

    print(f"Rows: {pf.metadata.num_rows:,}")
    print("Columns:", cols)

    preview = con_diag.execute(
        f"SELECT * FROM read_parquet({sqlq(path.resolve())}) LIMIT 5"
    ).fetchdf()
    display(preview)

    # Check candidate sample and row-index mapping columns.
    sample_col = next(
        (c for c in ["sample_id", "sample_index"] if c in cols), None
    )
    row_col = next(
        (c for c in ["row_index", "patch_row_index", "zarr_row_index"] if c in cols),
        None,
    )

    if sample_col and row_col:
        stats = con_diag.execute(f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(DISTINCT {qid(sample_col)}) AS unique_samples,
                COUNT(DISTINCT {qid(row_col)}) AS unique_row_indices,
                MIN({qid(row_col)}) AS min_row_index,
                MAX({qid(row_col)}) AS max_row_index,
                SUM(CASE WHEN {qid(row_col)} IS NULL THEN 1 ELSE 0 END)
                    AS missing_row_indices
            FROM read_parquet({sqlq(path.resolve())})
        """).fetchdf()

        print("Mapping statistics:")
        display(stats)

        duplicates = con_diag.execute(f"""
            SELECT {qid(sample_col)}, COUNT(*) AS n
            FROM read_parquet({sqlq(path.resolve())})
            GROUP BY {qid(sample_col)}
            HAVING COUNT(*) > 1
            LIMIT 10
        """).fetchdf()

        print("Duplicate sample IDs (first 10):")
        display(duplicates)

# 1. Inspect MODIS yearly sample-index files.
print("\n### MODIS SAMPLE INDEX FILES")

modis_sample_files = sorted(
    (ROOT / "modis").rglob("samples.parquet")
)

print("Files found:", len(modis_sample_files))

for path in modis_sample_files:
    inspect_mapping_parquet(path)

# 2. Inspect Sentinel-1 mapping/index Parquets wherever present.
print("\n### SENTINEL-1 SAMPLE / INDEX PARQUETS")

s1_root = ROOT / "sentinel1"
s1_candidates = sorted(
    p for p in s1_root.rglob("*.parquet")
    if any(term in p.name.lower() for term in (
        "sample", "index", "manifest", "mapping", "row"
    ))
)

print("Candidate files found:", len(s1_candidates))

for path in s1_candidates:
    inspect_mapping_parquet(path)

# 3. Inventory Zarr first-dimension lengths, without reading pixel values.
print("\n### ZARR ARRAY ROW COUNTS")

zarr_paths = sorted(
    p for root in [ROOT / "modis", s1_root]
    if root.exists()
    for p in root.rglob("*.zarr")
    if p.is_dir()
)

print("Zarr stores found:", len(zarr_paths))

for path in zarr_paths:
    print(f"\n{path.relative_to(ROOT)}")

    try:
        group = zarr.open_group(str(path), mode="r")

        def show_array(name, arr):
            if hasattr(arr, "shape") and hasattr(arr, "dtype"):
                shape = tuple(arr.shape)
                print(f"  {name}: shape={shape}, dtype={arr.dtype}")

        group.visititems(show_array)

    except Exception as exc:
        print("  Could not inspect:", repr(exc))

# 4. Look for potentially relevant SQLite indexing databases.
print("\n### SQLITE FILES")

sqlite_files = sorted(
    p for p in s1_root.rglob("*")
    if p.is_file()
    and (
        p.suffix.lower() in {".sqlite", ".db"}
        or p.name.lower().endswith(".sqlite3")
    )
)

for path in sqlite_files:
    print(path.relative_to(ROOT))

con_diag.close()


### MODIS SAMPLE INDEX FILES
Files found: 12

FILE: modis/evi_lst_patch_dataset/2015/samples.parquet
Rows: 329,903
Columns: ['sample_id', 'station_id', 'sample_date', 'modis_tile']


,sample_id,station_id,sample_date,modis_tile
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,h18v07
1,1371,AMMA-CATCH__Belefoungou-Mid,2015-04-01,h18v08
2,2316,AMMA-CATCH__Belefoungou-Top,2015-04-01,h18v08
3,4086,AMMA-CATCH__Nalohou-Top,2015-04-01,h18v08
4,4996,AMMA-CATCH__Tondikiboro,2015-04-01,h18v07



FILE: modis/evi_lst_patch_dataset/2016/samples.parquet
Rows: 394,790
Columns: ['sample_id', 'station_id', 'sample_date', 'modis_tile']


,sample_id,station_id,sample_date,modis_tile
0,275,AMMA-CATCH__Banizoumbou,2016-01-01,h18v07
1,1646,AMMA-CATCH__Belefoungou-Mid,2016-01-01,h18v08
2,2591,AMMA-CATCH__Belefoungou-Top,2016-01-01,h18v08
3,3374,AMMA-CATCH__Nalohou-Mid,2016-01-01,h18v08
4,4358,AMMA-CATCH__Nalohou-Top,2016-01-01,h18v08



FILE: modis/evi_lst_patch_dataset/2017/samples.parquet
Rows: 369,953
Columns: ['sample_id', 'station_id', 'sample_date', 'modis_tile']


,sample_id,station_id,sample_date,modis_tile
0,641,AMMA-CATCH__Banizoumbou,2017-01-01,h18v07
1,2007,AMMA-CATCH__Belefoungou-Mid,2017-01-01,h18v08
2,2957,AMMA-CATCH__Belefoungou-Top,2017-01-01,h18v08
3,3740,AMMA-CATCH__Nalohou-Mid,2017-01-01,h18v08
4,5637,AMMA-CATCH__Tondikiboro,2017-01-01,h18v07


... (99 outputs left out)

,row_count,unique_samples,unique_row_indices,min_row_index,max_row_index,missing_row_indices
0,394790,394790,394790,0,394789,0.0


Duplicate sample IDs (first 10):


,sample_id,n



### ZARR ARRAY ROW COUNTS
Zarr stores found: 24

modis/evi_lst_patch_dataset/2015/patches.zarr
  Could not inspect: AttributeError("'Group' object has no attribute 'visititems'")

modis/evi_lst_patch_dataset/2016/patches.zarr
  Could not inspect: AttributeError("'Group' object has no attribute 'visititems'")

modis/evi_lst_patch_dataset/2017/patches.zarr
  Could not inspect: AttributeError("'Group' object has no attribute 'visititems'")

modis/evi_lst_patch_dataset/2018/patches.zarr
  Could not inspect: AttributeError("'Group' object has no attribute 'visititems'")

modis/evi_lst_patch_dataset/2019/patches.zarr
  Could not inspect: AttributeError("'Group' object has no attribute 'visititems'")

modis/evi_lst_patch_dataset/2020/patches.zarr
  Could not inspect: AttributeError("'Group' object has no attribute 'visititems'")

modis/evi_lst_patch_dataset/2021/patches.zarr
  Could not inspect: AttributeError("'Group' object has no attribute 'visititems'")

modis/evi_lst_patch_dataset/2022/